最重要 字段校验、描述、嵌套结构、功能最丰富，类型不匹配的时候会抛出异常 ，返回的是Schema实例

运行时强制执行类型提示，确保数据的正确性和一致性，是生产场景首选

使用的时候，需要满足的3个条件 1 所有结构化输出的数据模型都必须继承BaseModel 2 使用类型提示Pydantic 支持丰富的字段类型str int float List[xxx] Optional[xxx] 4 使用Field()添加字段默认值和描述，帮助大模型理解字段含义

In [5]:
from pydantic import BaseModel, Field, ValidationError
from langchain_core.tools import tool
from langchain_core.utils.function_calling import convert_to_openai_tool
from rich import print as rprint
from typing import Literal, Optional
from langchain.chat_models import init_chat_model
from rich import print as rprint
##统一使用init_chat_model（）完成模型的初始化
import os
from dotenv import load_dotenv
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain_core.prompts import ChatPromptTemplate


#1 读取，env文件中的配置信息，相关的环境变量以.env文件中的优先
load_dotenv(override=True)
my_api_key=os.getenv("API_KEY")
my_base_url=os.getenv("BASE_URL")
#2 初始化大模型
model= init_chat_model(
    model="qwen3.8-max",
    model_provider="openai",
    api_key=my_api_key,
    base_url=my_base_url,
    timeout=30,
    max_retries=3,
)
message=[HumanMessage("我想知道刘亦菲的一些信息")]
class Person(BaseModel):
    """
    人物信息
    """
    name: str=Field(description="姓名",min_length=2,max_length=20)
    age:int=Field(description="年龄",gt=0,lt=120)
    occupation:str=Field(description="职业")
    hobby:str=Field(description="爱好")
    height:int=Field(description="身高",gt=0,lt=300)
    gender:str=Field(description="性别")
    weight:int=Field(description="体重",gt=0)
structured_model=model.with_structured_output(Person)

result=structured_model.invoke(message)
print(result)
# rprint(type(result))
# print(f"年龄:{result.age} 姓名:{result.name} 职业:{result.occupation}")

name='刘亦菲' age=37 occupation='演员' hobby='表演' height=170 gender='女' weight=45


高级特性 _1 可选字段 2 默认值(受大模型的供应商影响，有的支持，有的又不支持) 3 枚举类型 4 列表提取 5嵌套结构 6限制条件_

In [16]:
from pydantic import BaseModel, Field
from langchain.chat_models import init_chat_model
##统一使用init_chat_model（）完成模型的初始化
import os
from dotenv import load_dotenv
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from typing import Optional
#1 读取，env文件中的配置信息，相关的环境变量以.env文件中的优先
load_dotenv(override=True)
my_api_key=os.getenv("API_KEY")
my_base_url=os.getenv("BASE_URL")
#2 初始化大模型
model= init_chat_model(
    model="qwen3.8-max",
    model_provider="openai",
    api_key=my_api_key,
    base_url=my_base_url,
    timeout=30,
    max_retries=3,
)
message1=[HumanMessage("刘亦菲是1名演员")]
message2=[HumanMessage("张三是1名演员")]
class Person(BaseModel):
    """
    人物信息
    """
    name: str=Field(description="姓名",min_length=2,max_length=20)
    age:int=Field(default=20,description="年龄",gt=0,lt=120)
    occupation:str=Field(description="职业")
    hobby:Optional[str]=Field(description="爱好")
    height:int=Field(description="身高",gt=0,lt=300)
    gender:str=Field(description="性别")
    weight:int=Field(description="体重",gt=0)
class NormalPerson(BaseModel):
    """
    人物信息
    """
    name: str=Field(description="姓名",min_length=2,max_length=20)
    age:int=Field(default=10,description="年龄",gt=0,lt=120)
    occupation:str=Field(description="职业")

structured_model=model.with_structured_output(NormalPerson)

result=structured_model.invoke(message2)
print(result)
# rprint(type(result))
# print(f"年龄:{result.age} 姓名:{result.name} 职业:{result.occupation}")

name='John' age=25 occupation='engineer'


测试枚举值

In [20]:
from pydantic import BaseModel, Field
from langchain.chat_models import init_chat_model
##统一使用init_chat_model（）完成模型的初始化
import os
from dotenv import load_dotenv
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from typing import Optional
from enum import Enum
#1 读取，env文件中的配置信息，相关的环境变量以.env文件中的优先
load_dotenv(override=True)
my_api_key=os.getenv("API_KEY")
my_base_url=os.getenv("BASE_URL")
#2 初始化大模型
model= init_chat_model(
    model="qwen3.8-max",
    model_provider="openai",
    api_key=my_api_key,
    base_url=my_base_url,
    timeout=30,
    max_retries=3,
)

class Priority(str,Enum):
    low:str=Field(description="低")
    medium:str=Field(description="中")
    high:str=Field(description="高")

class Customer(BaseModel):
    """
    人物信息
    """
    name: str=Field(description="姓名",min_length=2,max_length=20)
    phone:str=Field(description="手机号",min_length=11,max_length=11)
    email:Optional[str]=Field(description="邮件")
    issue:str=Field(description="问题描述")
    #提供2种枚举操作，1是通过Enum来生成1个类，2是直接使用Literal
    # urgency:Priority=Field(description="紧急程度")
    urgency:Literal["低","中","高",]=Field(description="紧急程度")

structured_model=model.with_structured_output(Customer)
message=[HumanMessage(
    """
    客服：您好，请问有什么可以帮您？
    客户：我是王小明，电话18788483333，我的订单一直没发货，特别着急！！
    客服：好的，我马上帮您查询一下订单
    """
    )
]
result=structured_model.invoke(f"请您从以下客服对话中提取客户信息：\n{message}")
print(result)


name='王小明' phone='18788483333' email='' issue='订单一直没发货，特别着急' urgency='高'


列表提取的演示

In [21]:
from pydantic import BaseModel, Field
from langchain.chat_models import init_chat_model
##统一使用init_chat_model（）完成模型的初始化
import os
from dotenv import load_dotenv
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from typing import Optional
#1 读取，env文件中的配置信息，相关的环境变量以.env文件中的优先
load_dotenv(override=True)
my_api_key=os.getenv("API_KEY")
my_base_url=os.getenv("BASE_URL")
#2 初始化大模型
model= init_chat_model(
    model="qwen3.8-max",
    model_provider="openai",
    api_key=my_api_key,
    base_url=my_base_url,
    timeout=30,
    max_retries=3,
)
message=[HumanMessage("刘亦菲 30岁 电影演员，宋丹丹，76岁，小品演员")]


class Person(BaseModel):
    """
    人物信息
    """
    name: str=Field(description="姓名",min_length=2,max_length=20)
    age:int=Field(default=10,description="年龄",gt=0,lt=120)
    occupation:str=Field(description="职业")
class People(BaseModel):
    """
    人物的列表
    """
    people:list[Person]#多个Person的对象

structured_model=model.with_structured_output(People)

result=structured_model.invoke(message)
print(result)
# rprint(type(result))
# print(f"年龄:{result.age} 姓名:{result.name} 职业:{result.occupation}")

people=[Person(name='刘亦菲', age=30, occupation='电影演员'), Person(name='宋丹丹', age=76, occupation='小品演员')]


举例2 产品评论

In [22]:
class Review (BaseModel):
    product:str
    rating:int=Field(description="评分1-5")
    pros:list[str]=Field(description="优点列表")
    cons:list[str]=Field(description="缺点列表")
message=[HumanMessage(
    "iphone 17 很棒，摄像头强大，手感好，但是价格贵，没有配充电器，最多只能给4分"
)]
structured_model=model.with_structured_output(Review)

result=structured_model.invoke(message)
print(result)

product='iPhone 17' rating=4 pros=['摄像头强大', '手感好'] cons=['价格贵', '没有配充电器']


举例3 文档信息提取

In [23]:
class Invoice(BaseModel):
    invoiceNum:str=Field(description="发票号")
    date:str=Field(description="开票日期")
    total:float=Field(description="开票总额")
    items:list[str]=Field(description="商品列表")
message=[HumanMessage(
    """
    发票号：INV-2024-001
    日期：2024-01-15
    总金额：12999.00
    商品：MacBook Pro,Apple Care+
    """
)]
structured_model=model.with_structured_output(Invoice)

result=structured_model.invoke(f"提取下面的发票信息：{message}")
print(result)

invoiceNum='INV-2024-001' date='2024-01-15' total=12999.0 items=['MacBook Pro', 'Apple Care+']


嵌套结构的的演示 由于大模型的能力有限，负责嵌套结构可能会出错，最多3层

In [28]:
class Address(BaseModel):
    """
    地点的描述
    """
    city:str=Field(description="城市")
    district:str=Field(description="区、县")
    town:str=Field(description="镇")
class Company(BaseModel):
    """
    公司的描述
    """
    name:str=Field(description="公司名称")
    address:Address=Field(description="公司地址")
message=[HumanMessage("摸鱼有限公司在重庆市江北区观音镇")   ]
structured_model=model.with_structured_output(Company)

result=structured_model.invoke(message)
print(result)

name='查询企业' address=Address(city='重庆市', district='江北区', town='观音镇')


限制条件的演示 也和大模型的底层能力有关，有的大模型会过度的干涉用户的错误输入

In [33]:
from pydantic import ValidationError
class User(BaseModel):
    name:str=Field(description="姓名",min_length=2,max_length=20)
    age:int=Field(description="年龄",le=150,ge=0)
user1=User(name="张三",age=10)
#
# print(user1)
try:
    user2=User(name="刘三",age=160)
    print(f"OK user1:{user2}")
except ValidationError as e:
    print(f"Faild :{e}")

Faild :1 validation error for User
age
  Input should be less than or equal to 150 [type=less_than_equal, input_value=160, input_type=int]
    For further information visit https://errors.pydantic.dev/2.13/v/less_than_equal
